# 04 — Speech-to-Text with Whisper (Colab, T4 GPU)

**Project:** MeetingMind | **Phase:** 4

## What this notebook does
1. Sets up faster-whisper on a Colab T4
2. Runs our `TranscriptionService` (from `app/`) on real audio
3. Measures **WER** on LibriSpeech test-clean for two model sizes, plus speed and GPU memory
4. Runs the full path *video/audio -> ffmpeg -> Whisper -> timestamped JSON* on **your own file**
5. Tests the silence/hallucination failure mode and shows the effect of VAD

## Important limitation
LibriSpeech is clean **read** speech. A good WER here proves the integration is correct. It does **not**
predict WER on meetings (overlap, crosstalk, far-field mics). Meeting-domain WER on AMI comes in step 4b.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

## 1. Environment

In [1]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "NO GPU - change the runtime type!")

Tesla T4, 15360 MiB



In [2]:
!pip install -q faster-whisper jiwer pydantic-settings
!ffmpeg -version | head -1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.1 MB/s eta 0:00:00
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


## 2. Get the project code
Upload `meetingmind_phase4.zip` when prompted. (Alternative once your repo is on GitHub:
`!git clone https://github.com/<you>/meetingmind /content/meetingmind`.)

If you previously ran this notebook with an older zip, use **Runtime -> Restart session** first, otherwise Python keeps the old modules in memory.

In [3]:
import os, sys, zipfile
from pathlib import Path

import shutil

PROJECT_DIR = Path("/content/meetingmind")
FORCE_REUPLOAD = True   # True = always replace the code with the zip you upload (avoids stale code)

if FORCE_REUPLOAD or not (PROJECT_DIR / "app").exists():
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    from google.colab import files
    print("Upload meetingmind_phase4.zip")
    uploaded = files.upload()
    with zipfile.ZipFile(next(iter(uploaded))) as zf:
        zf.extractall("/content")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Working dir:", Path.cwd())
print(sorted(p.name for p in (PROJECT_DIR / "app").iterdir()))

Upload meetingmind_phase4.zip


Saving meetingmind_phase4 (1).zip to meetingmind_phase4 (1).zip
Working dir: /content/meetingmind
['__init__.py', 'core', 'models', 'services', 'utils']


## 3. Imports and configuration

In [4]:
import gc
import tarfile
import time

import jiwer
import numpy as np
import pandas as pd

from app.core.config import Settings
from app.services.transcription_service import TranscriptionService
from app.utils.audio_utils import extract_audio, get_duration_seconds
from app.utils.text_utils import normalize_for_wer

RANDOM_SEED = 42
N_EVAL_UTTERANCES = 100                     # subset of LibriSpeech test-clean (keeps runtime short)
MODELS_TO_COMPARE = ["small", "large-v3"]   # add "medium" if you want a middle point
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

def make_settings(model_size: str, **overrides) -> Settings:
    """T4 supports float16, so we use it explicitly instead of 'auto'."""
    return Settings(whisper_model_size=model_size, whisper_device="cuda",
                    whisper_compute_type="float16", data_dir=DATA_DIR, **overrides)

## 4. Dataset: LibriSpeech test-clean
- Source: https://www.openslr.org/12 | License: CC BY 4.0 | Download: about 350 MB
- Structure: `LibriSpeech/test-clean/<speaker>/<chapter>/` containing `.flac` files and one `<speaker>-<chapter>.trans.txt`
  per chapter, where each line is `<utterance-id> <UPPERCASE REFERENCE TEXT>`
- We evaluate a random, seeded subset (not the first N, which would all be a few speakers).

In [5]:
LIBRI_URL = "https://www.openslr.org/resources/12/test-clean.tar.gz"
LIBRI_DIR = DATA_DIR / "librispeech"
LIBRI_DIR.mkdir(exist_ok=True)
tar_path = LIBRI_DIR / "test-clean.tar.gz"

if not (LIBRI_DIR / "LibriSpeech" / "test-clean").exists():
    subprocess.run(["wget", "-q", "-c", LIBRI_URL, "-O", str(tar_path)], check=True)
    with tarfile.open(tar_path) as tf:
        tf.extractall(LIBRI_DIR)
    print("Downloaded and extracted.")

rows = []
for trans_file in sorted((LIBRI_DIR / "LibriSpeech" / "test-clean").rglob("*.trans.txt")):
    for line in trans_file.read_text().splitlines():
        utt_id, reference = line.split(" ", 1)
        rows.append({"utt_id": utt_id, "flac_path": trans_file.parent / f"{utt_id}.flac", "reference": reference})

manifest = pd.DataFrame(rows)
print("Utterances available:", len(manifest), "| speakers:", manifest["utt_id"].str.split("-").str[0].nunique())

eval_df = manifest.sample(N_EVAL_UTTERANCES, random_state=RANDOM_SEED).reset_index(drop=True)

# Convert through OUR audio utility, so the evaluation exercises the same path as production.
wav_dir = DATA_DIR / "librispeech_wav"
eval_df["wav_path"] = [extract_audio(p, wav_dir / f"{u}.wav") for p, u in zip(eval_df["flac_path"], eval_df["utt_id"])]
eval_df["duration"] = [get_duration_seconds(p) for p in eval_df["wav_path"]]
print(f"Eval subset: {len(eval_df)} utterances, {eval_df['duration'].sum() / 60:.1f} min of audio")
eval_df[["utt_id", "duration", "reference"]].head(3)

/tmp/ipykernel_806/588053654.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(LIBRI_DIR)


Downloaded and extracted.
Utterances available: 2620 | speakers: 40
Eval subset: 100 utterances, 12.2 min of audio


,utt_id,duration,reference
0,7729-102255-0008,19.065,ALL THE TERRITORIAL DIGNITARIES WERE PRESENT G...
1,1221-135766-0004,7.440,THIS OUTWARD MUTABILITY INDICATED AND DID NOT ...
2,4507-16021-0032,28.090,HE MUST DESCEND WITH HIS HEART FULL OF CHARITY...


## 5. Single-file smoke test
Look at the full `Transcript` object once, so you know exactly what downstream code will receive.
The first call is slower because CUDA kernels are warming up.

In [6]:
service = TranscriptionService(make_settings("small"))
sample = eval_df.iloc[0]
transcript = service.transcribe(sample["wav_path"])

print("Reference :", sample["reference"])
print("Hypothesis:", transcript.full_text)
print(f"\nlanguage={transcript.language} (p={transcript.language_probability:.2f}) | "
      f"audio={transcript.audio_duration:.1f}s | processing={transcript.processing_seconds:.2f}s | "
      f"RTF={transcript.real_time_factor:.3f}")
print("\nFirst segment, word by word:")
for w in transcript.segments[0].words[:10]:
    print(f"  {w.start:6.2f} -> {w.end:6.2f}  p={w.probability:.2f}  {w.text}")
del service; gc.collect()

Reference : ALL THE TERRITORIAL DIGNITARIES WERE PRESENT GOVERNOR SHANNON PRESIDED JOHN CALHOUN THE SURVEYOR GENERAL MADE THE PRINCIPAL SPEECH A DENUNCIATION OF THE ABOLITIONISTS SUPPORTING THE TOPEKA MOVEMENT CHIEF JUSTICE LECOMPTE DIGNIFIED THE OCCASION WITH APPROVING REMARKS
Hypothesis: All the territorial dignitaries were present. Governor Shannon presided. John Calhoun, the surveyor general, made the principal speech, a denunciation of the abolitionist, supporting the Topeka movement. Chief Justice Lacombe dignified the occasion with approving remarks.

language=en (p=1.00) | audio=19.1s | processing=1.70s | RTF=0.089

First segment, word by word:
    0.00 ->   0.36  p=0.74  All
    0.36 ->   0.52  p=0.97  the
    0.52 ->   1.12  p=0.99  territorial
    1.12 ->   2.02  p=1.00  dignitaries
    2.02 ->   2.16  p=1.00  were
    2.16 ->   2.56  p=1.00  present.


88

## 6. WER
**Word Error Rate** = (S + D + I) / N, where S = substituted words, D = deleted, I = inserted, N = words in the reference.
Computed from a word-level edit-distance alignment. It can exceed 100% (many insertions).

Rules we follow:
- **Corpus-level WER**: total errors / total reference words. Do **not** average per-utterance WERs, because short utterances would be over-weighted.
- **Normalize both sides** with `normalize_for_wer` (lowercase, strip punctuation). Otherwise Whisper's capitalisation and punctuation count as errors.
- Known remaining mismatch: digits vs spelled-out numbers ("10" vs "TEN"), which inflates our WER slightly relative to published numbers that use a fuller normalizer.

In [7]:
def corpus_wer(references: list[str], hypotheses: list[str]) -> dict:
    refs = [normalize_for_wer(r) for r in references]
    hyps = [normalize_for_wer(h) for h in hypotheses]
    out = jiwer.process_words(refs, hyps)
    n_ref = out.hits + out.substitutions + out.deletions
    return {"wer": out.wer, "substitutions": out.substitutions, "deletions": out.deletions,
            "insertions": out.insertions, "ref_words": n_ref}


def gpu_memory_mb() -> int:
    result = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                            capture_output=True, text=True)
    return int(result.stdout.strip().splitlines()[0])


def evaluate_model(model_size: str, data: pd.DataFrame) -> tuple[dict, list[str]]:
    service = TranscriptionService(make_settings(model_size))
    t0 = time.perf_counter()
    service.transcribe(data.iloc[0]["wav_path"])            # warm-up (loads weights, initialises CUDA)
    load_and_warmup = time.perf_counter() - t0

    hypotheses, processing, audio = [], 0.0, 0.0
    for row in data.itertuples():
        t = service.transcribe(row.wav_path)
        hypotheses.append(t.full_text)
        processing += t.processing_seconds
        audio += t.audio_duration

    memory = gpu_memory_mb()                                 # model still loaded here; includes CUDA context
    metrics = {"model": model_size, **corpus_wer(data["reference"].tolist(), hypotheses),
               "audio_min": audio / 60, "processing_s": processing, "RTF": processing / audio,
               "gpu_mem_mb": memory, "load_warmup_s": load_and_warmup}
    del service; gc.collect()
    return metrics, hypotheses

## 7. Compare model sizes (a few minutes per model)

In [8]:
results, all_hypotheses = [], {}
for size in MODELS_TO_COMPARE:
    metrics, hyps = evaluate_model(size, eval_df)
    results.append(metrics)
    all_hypotheses[size] = hyps
    print(f"{size}: WER={metrics['wer'] * 100:.2f}%  RTF={metrics['RTF']:.3f}  GPU={metrics['gpu_mem_mb']} MB")

results_df = pd.DataFrame(results)
results_df["wer_%"] = (results_df["wer"] * 100).round(2)
results_df.round(3)

small: WER=4.70%  RTF=0.046  GPU=695 MB
large-v3: WER=5.38%  RTF=0.183  GPU=3223 MB


,model,wer,substitutions,deletions,insertions,ref_words,audio_min,processing_s,RTF,gpu_mem_mb,load_warmup_s,wer_%
0,small,0.047,70,9,10,1895,12.198,33.689,0.046,695,1.293,4.70
1,large-v3,0.054,44,5,53,1895,12.198,133.831,0.183,3223,24.262,5.38


### Reading the table
- **WER** should be low single digits for clean read speech. If it is above ~10%, something is wrong (wrong audio path, wrong normalisation, wrong language).
- **RTF** (real-time factor) = processing time / audio time. 0.05 means 20x faster than real time. A 1-hour meeting costs roughly `RTF x 60` minutes.
- **S / D / I** split tells you the error type. Whisper errors are mostly substitutions (spelling, homophones, numbers). A high deletion count would suggest VAD cut real speech.
- `gpu_mem_mb` comes from `nvidia-smi`, so it includes the CUDA context and anything else on the GPU. Treat it as an upper-bound estimate.
- With only 100 utterances (see `ref_words` for the total word count) the WER has real sampling uncertainty. Do not over-read a 0.3-point difference between models.

## 8. Error analysis
Look at the actual mistakes of the better model before trusting a single number.

In [9]:
best_model = results_df.sort_values("wer").iloc[0]["model"]
print("Best model:", best_model)

refs = [normalize_for_wer(r) for r in eval_df["reference"]]
hyps = [normalize_for_wer(h) for h in all_hypotheses[best_model]]
per_utt = pd.DataFrame({
    "utt_id": eval_df["utt_id"],
    "wer": [jiwer.wer(r, h) for r, h in zip(refs, hyps)],
    "ref": refs, "hyp": hyps,
})
print(f"Utterances with zero errors: {(per_utt['wer'] == 0).mean() * 100:.0f}%")

pd.set_option("display.max_colwidth", 200)
for _, r in per_utt.sort_values("wer", ascending=False).head(8).iterrows():
    print(f"\n[{r['utt_id']}] WER={r['wer']:.2f}\n  REF: {r['ref']}\n  HYP: {r['hyp']}")

Best model: small
Utterances with zero errors: 52%

[2961-961-0003] WER=0.71
  REF: i will if timaeus approves i approve
  HYP: i will if to me as a proof i approve

[8555-284449-0014] WER=0.50
  REF: the former boolooroo groaned
  HYP: the former boolaroo growned

[2094-142345-0017] WER=0.29
  REF: i never knew your equals for gallowsness
  HYP: i never knew your equals for gallows ness

[237-134493-0010] WER=0.29
  REF: i never see lou's scythe over here
  HYP: i never see loose side over here

[4446-2273-0011] WER=0.25
  REF: there is nothing else that looks so jolly
  HYP: there's nothing else that looks so jolly

[908-157963-0028] WER=0.25
  REF: or an eye of gifts and graces showring fruits and coined gold
  HYP: or an eye of gifts and graces showering fruits and coin at gold

[237-134500-0021] WER=0.23
  REF: oh ever so much only he seems kind of staid and school teachery
  HYP: oh ever so much only he seems kind of stay it in school teachery

[6829-68769-0048] WER=0.23
  REF: g

## 9. Your own file: video or audio -> timestamped transcript
Upload a **short** meeting recording (audio or video, 1-10 minutes is ideal). This is the real pipeline:
`ffmpeg extraction -> Whisper -> Transcript JSON`.

Segments are flagged using Whisper's own default thresholds:
- `avg_logprob < -1.0`: low-confidence decoding
- `no_speech_prob > 0.6`: probably not speech
- `compression_ratio > 2.4`: highly repetitive text, a typical hallucination loop

A flag means "check this", not "this is wrong".

In [10]:
from google.colab import files

MEETING_MODEL = "small"   # try "large-v3" and compare the output yourself

def fmt(seconds: float) -> str:
    return f"{int(seconds // 60):02d}:{seconds % 60:05.2f}"

def flags(seg) -> str:
    out = []
    if seg.avg_logprob is not None and seg.avg_logprob < -1.0: out.append("LOW-CONF")
    if seg.no_speech_prob is not None and seg.no_speech_prob > 0.6: out.append("NO-SPEECH?")
    if seg.compression_ratio is not None and seg.compression_ratio > 2.4: out.append("REPETITIVE")
    return ",".join(out)

uploaded = files.upload()
user_file = DATA_DIR / "user_upload" / next(iter(uploaded))
user_file.parent.mkdir(exist_ok=True)
user_file.write_bytes(uploaded[next(iter(uploaded))])

wav = extract_audio(user_file, DATA_DIR / "user_audio" / f"{user_file.stem}.wav")
print(f"Extracted: {wav.name} | duration {get_duration_seconds(wav) / 60:.1f} min")

service = TranscriptionService(make_settings(MEETING_MODEL))
user_transcript = service.transcribe(wav)
print(f"language={user_transcript.language} (p={user_transcript.language_probability:.2f}) | "
      f"processing={user_transcript.processing_seconds:.1f}s | RTF={user_transcript.real_time_factor:.3f} | "
      f"segments={len(user_transcript.segments)}")

for seg in user_transcript.segments[:40]:
    print(f"[{fmt(seg.start)} - {fmt(seg.end)}] {seg.text}  {('<' + flags(seg) + '>') if flags(seg) else ''}")

n_flagged = sum(bool(flags(s)) for s in user_transcript.segments)
print(f"\nFlagged segments: {n_flagged}/{len(user_transcript.segments)}")

json_path = user_transcript.save_json(DATA_DIR / "transcripts" / f"{user_file.stem}.{MEETING_MODEL}.json")
files.download(str(json_path))

Saving 09-15-2026-Council-Meeting.mp3 to 09-15-2026-Council-Meeting.mp3
Extracted: 09-15-2026-Council-Meeting.wav | duration 10.1 min
language=en (p=0.99) | processing=13.5s | RTF=0.022 | segments=85
[00:01.90 - 00:15.16] Thank you very much. This is the arm of Springfield meeting agenda for September the 15th of 2026. We're calling the meeting to order exactly 6pm. All of council is present here.  
[00:15.88 - 00:30.94] Our CEO is is on a course our assistant CEO, Christie grown height is in the position here. I'll present council and descending order here to my right and descending as deputy mayor and council Glenn Glenn fuel.  
[00:32.54 - 00:37.10] councilors Kaczynski, Miller and Warren. Then we'll go to land acknowledgement  
[00:37.10 - 00:42.14] please. That's item number three on the agenda of people are following. The  
[00:42.14 - 00:46.16] Armour Springfield acknowledges that we are gathered on ancestral lands, 3D1  
[00:46.16 - 00:51.10] territory, traditional territory in

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Experiment: does Whisper hallucinate on non-speech, and does VAD help?
Whisper always tries to output text. On silence or noise it can invent phrases. We feed 60 s of quiet noise
with VAD **off** and **on**. It may or may not hallucinate with this particular signal, so report whatever you see.
Real recordings (room tone, music, long pauses) are the real risk.

In [11]:
import wave

noise_path = DATA_DIR / "noise_60s.wav"
rng = np.random.default_rng(0)
noise = (rng.normal(0, 0.01, 16_000 * 60) * 32767).astype(np.int16)
with wave.open(str(noise_path), "wb") as wf:
    wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(16_000)
    wf.writeframes(noise.tobytes())

for vad in (False, True):
    svc = TranscriptionService(make_settings("small", whisper_vad_filter=vad))
    t = svc.transcribe(noise_path)
    print(f"VAD={vad!s:<5} -> {len(t.segments)} segment(s): {[s.text for s in t.segments][:5]}")
    del svc; gc.collect()

VAD=False -> 1 segment(s): ['ᶦᶦᶦᶦᶦᶦᶦᶦᶦᶦᶦ']
VAD=True  -> 0 segment(s): []


## 11. Findings summary (paste this back)

In [12]:
import faster_whisper, ctranslate2

print("=== FINDINGS TO REPORT ===")
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip())
print("faster-whisper:", faster_whisper.__version__, "| ctranslate2:", ctranslate2.__version__)
print(results_df[["model", "wer_%", "substitutions", "deletions", "insertions", "ref_words",
                  "RTF", "gpu_mem_mb", "load_warmup_s"]].to_string(index=False))
print(f"Own file ({MEETING_MODEL}): RTF={user_transcript.real_time_factor:.3f}, "
      f"segments={len(user_transcript.segments)}, flagged={n_flagged}")
print("ALSO PASTE: 15-20 lines of your own-file transcript, and the section 10 output.")
results_df.to_csv(DATA_DIR / "asr_librispeech_results.csv", index=False)
files.download(str(DATA_DIR / "asr_librispeech_results.csv"))

=== FINDINGS TO REPORT ===
GPU: Tesla T4
faster-whisper: 1.2.1 | ctranslate2: 4.8.2
   model  wer_%  substitutions  deletions  insertions  ref_words      RTF  gpu_mem_mb  load_warmup_s
   small   4.70             70          9          10       1895 0.046031         695       1.292957
large-v3   5.38             44          5          53       1895 0.182861        3223      24.261725
Own file (small): RTF=0.022, segments=85, flagged=0
ALSO PASTE: 15-20 lines of your own-file transcript, and the section 10 output.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next
- **4b:** meeting-domain WER on AMI (overlapping speech, far-field), comparing with LibriSpeech
- **Phase 5:** speaker diarization and merging speaker labels onto these word timestamps